# Desarrollo de un Sistema de Generación Aumentada por Recuperación (RAG) para la Consulta y Auditoría de Normativas Internacionales de Inteligencia Artificial.
## Construcción del Índice Vectorial y Recuperación Híbrida

**Autor:** Eduard David Apolo Gallardo  
**Máster en Deep Learning — UPM**

---
Se implementan y comparan dos modelos de embeddings y tres estrategias de recuperación:

| Componente | Tecnología |
|---|---|
| Embedding baseline | `paraphrase-multilingual-MiniLM-L12-v2` |
| Embedding mejorado | `BAAI/bge-m3` |
| Base de datos vectorial | ChromaDB (persistencia en disco) |
| Recuperación léxica | BM25 (rank-bm25) |
| Fusión de rankings | Reciprocal Rank Fusion (RRF) |

---
## Importaciones

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json
import time
import logging
import random
import numpy as np
import torch
import chromadb
import unicodedata
import pickle
from rank_bm25 import BM25Okapi
import re
import gc
from chromadb.config import Settings
from pathlib import Path
from langchain_community.docstore.document import Document
from sentence_transformers import SentenceTransformer
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

def purgar_vram_gpu(variables_a_eliminar: list = None) -> None:
    if variables_a_eliminar is not None:
        for var in variables_a_eliminar:
            try:
                del var
            except NameError:
                pass
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()
        
        print("Purga de VRAM completada. Memoria reservada actual: "
              f"{torch.cuda.memory_reserved(0) / (1024**2):.2f} MB")
    else:
        print("No se detectó un dispositivo CUDA activo.")

# SEMILLAS PARA REPRODUCIBILIDAD
random.seed(42)
np.random.seed(42)
DetectorFactory.seed = 42
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

os.environ["HF_HUB_DISABLE_XET"] = "1"
BASE_DIR    = Path(r"E:\Proyectos_IA\Deep_Learning\TFM")
DATA_DIR    = BASE_DIR / "Data"
VECTOR_DIR  = BASE_DIR / "VectorDB"
LOG_DIR     = BASE_DIR / "Logs"

# Directorios de ChromaDB por modelo de embeddings
CHROMA_BASELINE_DIR = VECTOR_DIR / "chroma_baseline"
CHROMA_BGE_M3_DIR   = VECTOR_DIR / "chroma_bge_m3"

for directorio in [VECTOR_DIR, CHROMA_BASELINE_DIR, CHROMA_BGE_M3_DIR, LOG_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

# CONFIGURACIÓN DE LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_DIR / "fase2_vectorizacion.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# VERIFICACIÓN DE GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    logger.info(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    logger.warning("GPU no detectada. Se utilizará CPU.")

# CARGA DEL CORPUS PROCESADO
RUTA_CORPUS = DATA_DIR / "corpus_tfm_procesado.json"

with open(RUTA_CORPUS, "r", encoding="utf-8") as f:
    corpus_raw = json.load(f)

# Reconstrucción como objeto LangChain
fragmentos = [
    Document(
        page_content=item["contenido"],
        metadata=item["metadatos"]
    )
    for item in corpus_raw
]

logger.info(f"Corpus cargado: {len(fragmentos)} fragmentos desde '{RUTA_CORPUS.name}'")
print(f"\nEjemplo de fragmento cargado:")
print(f"  chunk_id   : {fragmentos[0].metadata['chunk_id']}")
print(f"  source     : {fragmentos[0].metadata['source']}")
print(f"  longitud   : {fragmentos[0].metadata['longitud_chars']} chars")
print(f"  contenido  : {fragmentos[0].page_content[:150]}")

C:\Users\a3dua\AppData\Local\Temp\ipykernel_40912\3661620667.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore.document import Document
2026-06-13 15:01:30,155 [INFO] GPU detectada: NVIDIA GeForce RTX 3050 Ti Laptop GPU
2026-06-13 15:01:30,155 [INFO] VRAM disponible: 4.00 GB
2026-06-13 15:01:30,266 [INFO] Corpus cargado: 5192 fragmentos desde 'corpus_tfm_procesado.json'



Ejemplo de fragmento cargado:
  chunk_id   : 0
  source     : AEPD EIPD/gestion-riesgo-y-evaluacion-impacto-en-tratamientos-datos-personales.pdf
  longitud   : 217 chars
  contenido  : Gestión del riesgo y 
evaluación de impacto 
en tratamientos de datos 
personales 
 
 
 
 
 
 
Junio 2021 
 
Esta obra está bajo una 
Licencia Creativ


---
## Modelos de embeddings

Se cargan ambos modelos y se comparan sus dimensiones de salida y tiempos de
codificación sobre una frase de prueba del dominio normativo.

In [2]:
MODELO_BASELINE_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_BGE_M3_ID   = "BAAI/bge-m3"

logger.info("Cargando modelo baseline (MiniLM-L12)...")
t0 = time.time()
modelo_baseline = SentenceTransformer(MODELO_BASELINE_ID, device=device)
logger.info(f"Modelo baseline cargado en {time.time() - t0:.2f}s")

logger.info("Cargando modelo mejorado (bge-m3)...")
t0 = time.time()
modelo_bge_m3 = SentenceTransformer(
    MODELO_BGE_M3_ID,
    device=device,
    model_kwargs={"torch_dtype": torch.float16}
)
logger.info(f"Modelo bge-m3 cargado en {time.time() - t0:.2f}s")

frase_prueba = (
    "¿Cuáles son las obligaciones del proveedor de un sistema de IA "
    "de alto riesgo según el Artículo 16 del AI Act?"
)

t0 = time.time()
emb_baseline = modelo_baseline.encode(frase_prueba, normalize_embeddings=True)
t_baseline = time.time() - t0

t0 = time.time()
emb_bge_m3 = modelo_bge_m3.encode(frase_prueba, normalize_embeddings=True)
t_bge_m3 = time.time() - t0

print("\n" + "=" * 60)
print("COMPARATIVA DE MODELOS DE EMBEDDINGS")
print("=" * 60)
print(f"{'Propiedad':<30} {'Baseline (MiniLM)':<22} {'Mejorado (bge-m3)'}")
print("-" * 60)
print(f"{'Dimensión del vector':<30} {emb_baseline.shape[0]:<22} {emb_bge_m3.shape[0]}")
print(f"{'Tiempo de codificación (1 frase)':<30} {t_baseline*1000:<22.1f} {t_bge_m3*1000:.1f} ms")
print(f"{'Tipo de dato':<30} {str(emb_baseline.dtype):<22} {str(emb_bge_m3.dtype)}")
print(f"{'Normalización L2':<30} {'Sí':<22} {'Sí'}")
print("=" * 60)

2026-06-13 15:01:36,264 [INFO] Cargando modelo baseline (MiniLM-L12)...
2026-06-13 15:01:36,975 [INFO] Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.
2026-06-13 15:01:41,124 [INFO] Modelo baseline cargado en 4.86s
2026-06-13 15:01:41,125 [INFO] Cargando modelo mejorado (bge-m3)...
2026-06-13 15:01:41,519 [INFO] Loading SentenceTransformer model from BAAI/bge-m3.
2026-06-13 15:01:47,803 [INFO] Modelo bge-m3 cargado en 6.68s


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


COMPARATIVA DE MODELOS DE EMBEDDINGS
Propiedad                      Baseline (MiniLM)      Mejorado (bge-m3)
------------------------------------------------------------
Dimensión del vector           384                    1024
Tiempo de codificación (1 frase) 225.7                  102.5 ms
Tipo de dato                   float32                float16
Normalización L2               Sí                     Sí


---
## Vectorización del corpus con ambos modelos

In [3]:
textos_corpus = [f.page_content for f in fragmentos]
BATCH_SIZE_BASELINE = 64
BATCH_SIZE_BGE_M3   = 32

# VECTORIZACIÓN CON MODELO BASELINE
logger.info("Iniciando vectorización con modelo baseline (MiniLM-L12)...")
t0 = time.time()

embeddings_baseline = modelo_baseline.encode(
    textos_corpus,
    batch_size=BATCH_SIZE_BASELINE,
    normalize_embeddings=True,
    show_progress_bar=True,
    device=device
)

t_baseline_total = time.time() - t0
logger.info(
    f"Vectorización baseline completada: {len(embeddings_baseline)} vectores "
    f"de dimensión {embeddings_baseline.shape[1]} en {t_baseline_total:.1f}s"
)

# Liberación de caché GPU antes de cargar el siguiente modelo
purgar_vram_gpu()

# VECTORIZACIÓN CON MODELO bge-m3
logger.info("Iniciando vectorización con modelo mejorado (bge-m3)...")
t0 = time.time()

embeddings_bge_m3 = modelo_bge_m3.encode(
    textos_corpus,
    batch_size=BATCH_SIZE_BGE_M3,
    normalize_embeddings=True,
    show_progress_bar=True,
    device=device
)

t_bge_m3_total = time.time() - t0
logger.info(
    f"Vectorización bge-m3 completada: {len(embeddings_bge_m3)} vectores "
    f"de dimensión {embeddings_bge_m3.shape[1]} en {t_bge_m3_total:.1f}s"
)

purgar_vram_gpu()

print("\n" + "=" * 60)
print("VECTORIZACIÓN")
print("=" * 60)
print(f"{'Modelo':<30} {'Vectores':<12} {'Dimensión':<12} {'Tiempo'}")
print("-" * 60)
print(f"{'MiniLM-L12 (baseline)':<30} {len(embeddings_baseline):<12} "
      f"{embeddings_baseline.shape[1]:<12} {t_baseline_total:.1f}s")
print(f"{'bge-m3 (mejorado)':<30} {len(embeddings_bge_m3):<12} "
      f"{embeddings_bge_m3.shape[1]:<12} {t_bge_m3_total:.1f}s")
print("=" * 60)
print(f"\nVelocidad baseline : {len(textos_corpus)/t_baseline_total:.0f} fragmentos/seg")
print(f"Velocidad bge-m3   : {len(textos_corpus)/t_bge_m3_total:.0f} fragmentos/seg")

2026-06-13 15:01:57,097 [INFO] Iniciando vectorización con modelo baseline (MiniLM-L12)...


Batches:   0%|          | 0/82 [00:00<?, ?it/s]

2026-06-13 15:02:07,464 [INFO] Vectorización baseline completada: 5192 vectores de dimensión 384 en 10.4s
2026-06-13 15:02:07,598 [INFO] Iniciando vectorización con modelo mejorado (bge-m3)...


Purga de VRAM completada. Memoria reservada actual: 1570.00 MB


Batches:   0%|          | 0/163 [00:00<?, ?it/s]

2026-06-13 15:03:21,329 [INFO] Vectorización bge-m3 completada: 5192 vectores de dimensión 1024 en 73.7s


Purga de VRAM completada. Memoria reservada actual: 1570.00 MB

VECTORIZACIÓN
Modelo                         Vectores     Dimensión    Tiempo
------------------------------------------------------------
MiniLM-L12 (baseline)          5192         384          10.4s
bge-m3 (mejorado)              5192         1024         73.7s

Velocidad baseline : 501 fragmentos/seg
Velocidad bge-m3   : 70 fragmentos/seg


---
## ChromaDB

In [4]:
def crear_coleccion_chroma(
    directorio: Path,
    nombre_coleccion: str,
    fragmentos: list[Document],
    embeddings: np.ndarray,
    batch_size: int = 500
) -> chromadb.Collection:
    cliente = chromadb.PersistentClient(
        path=str(directorio),
        settings=Settings(anonymized_telemetry=False)
    )

    try:
        cliente.delete_collection(nombre_coleccion)
        logger.info(f"Colección previa '{nombre_coleccion}' eliminada.")
    except Exception:
        pass

    coleccion = cliente.create_collection(
        name=nombre_coleccion,
        metadata={"hnsw:space": "cosine"}
    )

    # Preparación de metadatos serializables para ChromaDB
    def serializar_metadatos(meta: dict) -> dict:
        meta_limpio = {}
        for k, v in meta.items():
            if isinstance(v, list):
                meta_limpio[k] = ", ".join(str(x) for x in v) if v else ""
            elif v is None:
                meta_limpio[k] = ""
            elif isinstance(v, (str, int, float, bool)):
                meta_limpio[k] = v
            else:
                meta_limpio[k] = str(v)
        return meta_limpio

    # Inserción por lotes
    total = len(fragmentos)
    for inicio in range(0, total, batch_size):
        fin = min(inicio + batch_size, total)
        lote_fragmentos = fragmentos[inicio:fin]
        lote_embeddings = embeddings[inicio:fin]

        coleccion.add(
            ids=[str(f.metadata["chunk_id"]) for f in lote_fragmentos],
            embeddings=lote_embeddings.tolist(),
            documents=[f.page_content for f in lote_fragmentos],
            metadatas=[serializar_metadatos(f.metadata) for f in lote_fragmentos]
        )
        logger.info(f"  Insertados fragmentos {inicio}–{fin} / {total}")

    logger.info(f"Colección '{nombre_coleccion}' creada con {coleccion.count()} vectores.")
    return coleccion

# CREACIÓN DE ÍNDICE BASELINE (MiniLM-L12)
logger.info("Creando índice ChromaDB — modelo baseline (MiniLM-L12)...")
t0 = time.time()

coleccion_baseline = crear_coleccion_chroma(
    directorio=CHROMA_BASELINE_DIR,
    nombre_coleccion="ai_act_baseline",
    fragmentos=fragmentos,
    embeddings=embeddings_baseline
)

logger.info(f"Índice baseline creado en {time.time()-t0:.1f}s")

# CREACIÓN DE ÍNDICE bge-m3
logger.info("Creando índice ChromaDB — modelo mejorado (bge-m3)...")
t0 = time.time()

coleccion_bge_m3 = crear_coleccion_chroma(
    directorio=CHROMA_BGE_M3_DIR,
    nombre_coleccion="ai_act_bge_m3",
    fragmentos=fragmentos,
    embeddings=embeddings_bge_m3
)

logger.info(f"Índice bge-m3 creado en {time.time()-t0:.1f}s")

print("\n" + "=" * 55)
print("ÍNDICES CHROMADB CREADOS")
print("=" * 55)
print(f"  Baseline — vectores indexados : {coleccion_baseline.count()}")
print(f"  bge-m3  — vectores indexados  : {coleccion_bge_m3.count()}")
print(f"  Persistencia baseline         : {CHROMA_BASELINE_DIR}")
print(f"  Persistencia bge-m3           : {CHROMA_BGE_M3_DIR}")
print("=" * 55)

2026-06-13 15:03:58,209 [INFO] Creando índice ChromaDB — modelo baseline (MiniLM-L12)...
2026-06-13 15:04:02,214 [INFO] Colección previa 'ai_act_baseline' eliminada.
2026-06-13 15:04:02,977 [INFO]   Insertados fragmentos 0–500 / 5192
2026-06-13 15:04:03,567 [INFO]   Insertados fragmentos 500–1000 / 5192
2026-06-13 15:04:04,381 [INFO]   Insertados fragmentos 1000–1500 / 5192
2026-06-13 15:04:05,067 [INFO]   Insertados fragmentos 1500–2000 / 5192
2026-06-13 15:04:05,809 [INFO]   Insertados fragmentos 2000–2500 / 5192
2026-06-13 15:04:06,461 [INFO]   Insertados fragmentos 2500–3000 / 5192
2026-06-13 15:04:07,041 [INFO]   Insertados fragmentos 3000–3500 / 5192
2026-06-13 15:04:07,652 [INFO]   Insertados fragmentos 3500–4000 / 5192
2026-06-13 15:04:08,586 [INFO]   Insertados fragmentos 4000–4500 / 5192
2026-06-13 15:04:09,593 [INFO]   Insertados fragmentos 4500–5000 / 5192
2026-06-13 15:04:09,919 [INFO]   Insertados fragmentos 5000–5192 / 5192
2026-06-13 15:04:09,932 [INFO] Colección 'ai_ac


ÍNDICES CHROMADB CREADOS
  Baseline — vectores indexados : 5192
  bge-m3  — vectores indexados  : 5192
  Persistencia baseline         : E:\Proyectos_IA\Deep_Learning\TFM\VectorDB\chroma_baseline
  Persistencia bge-m3           : E:\Proyectos_IA\Deep_Learning\TFM\VectorDB\chroma_bge_m3


---
## Verificación semántica

Se lanza una consulta de prueba sobre ambos índices y se comparan los resultados
recuperados. Se verifica que los fragmentos retornados son normativamente relevantes.

In [5]:
mapa_chunk_id = {f.metadata["chunk_id"]: i for i, f in enumerate(fragmentos)}

def busqueda_semantica(
    consulta: str,
    modelo: SentenceTransformer,
    coleccion: chromadb.Collection,
    k: int = 10,
    filtrar_idioma: str = "es"
) -> list[dict]:
    vector = modelo.encode(consulta, normalize_embeddings=True)

    kwargs = {
        "query_embeddings" : [vector.tolist()],
        "n_results"        : k,
        "include"          : ["documents", "metadatas", "distances"]
    }
    # Filtro de idioma aplicado como condición de metadatos en ChromaDB
    if filtrar_idioma:
        kwargs["where"] = {"idioma": filtrar_idioma}

    resultados = coleccion.query(**kwargs)

    return [
        {
            "contenido" : resultados["documents"][0][i],
            "metadatos" : resultados["metadatas"][0][i],
            "distancia" : resultados["distances"][0][i],
            "indice"    : mapa_chunk_id.get(
                int(resultados["metadatas"][0][i].get("chunk_id", -1)), -1
            ),
            "ranking"   : i + 1
        }
        for i in range(len(resultados["documents"][0]))
    ]

# CONSULTA DE PRUEBA
CONSULTA_PRUEBA = (
    "¿Cuáles son las obligaciones del proveedor de un sistema de IA "
    "de alto riesgo según el AI Act?"
)
K_RESULTADOS = 5

print(f"Consulta: '{CONSULTA_PRUEBA}'")
print(f"K = {K_RESULTADOS}\n")

# Búsqueda con modelo baseline
resultados_baseline = busqueda_semantica(
    CONSULTA_PRUEBA, modelo_baseline, coleccion_baseline, K_RESULTADOS
)

# Búsqueda con modelo bge-m3
resultados_bge_m3 = busqueda_semantica(
    CONSULTA_PRUEBA, modelo_bge_m3, coleccion_bge_m3, K_RESULTADOS
)

def imprimir_resultados(resultados: list[dict], nombre_modelo: str):
    print(f"\n{'='*60}")
    print(f"RESULTADOS — {nombre_modelo}")
    print(f"{'='*60}")
    for r in resultados:
        articulos = r['metadatos'].get('articulos', 'N/A')
        fuente    = r['metadatos'].get('source', 'N/A')
        pagina    = r['metadatos'].get('page', 'N/A')
        print(f"\n  [{r['ranking']}] Distancia: {r['distancia']:.4f}")
        print(f"       Fuente   : {fuente}")
        print(f"       Artículos: {articulos}")
        print(f"       Página   : {pagina}")
        print(f"       Contenido: {r['contenido'][:200]}...")

imprimir_resultados(resultados_baseline, "MiniLM-L12 (Baseline)")
imprimir_resultados(resultados_bge_m3,   "bge-m3 (Mejorado)")

Consulta: '¿Cuáles son las obligaciones del proveedor de un sistema de IA de alto riesgo según el AI Act?'
K = 5



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


RESULTADOS — MiniLM-L12 (Baseline)

  [1] Distancia: 0.1728
       Fuente   : AESIA/11-guia-ciberseguridad.pdf
       Artículos: 
       Página   : 49
       Contenido: 49 
 
Si el formato de uso y comercialización del sistema de IA contemplase que el responsable 
del despliegue dispusiese del sistema de IA como un activo de sus propias instalaciones 
(ya sea on -pre...

  [2] Distancia: 0.1843
       Fuente   : AESIA/05-guia-de-gestion-de-riesgos.pdf
       Artículos: 9
       Página   : 11
       Contenido: 11 
 
3. Reglamento de Inteligencia 
Artificial 
La puesta en servicio o la utilización de sistemas de IA de alto riesgo debe supeditarse 
al cumplimiento de determinados requisitos obligatorios, entr...

  [3] Distancia: 0.1927
       Fuente   : AESIA/11-guia-ciberseguridad.pdf
       Artículos: 
       Página   : 6
       Contenido: 1.3 ¿A quién está dirigido? 
Es responsabilidad del proveedor del sistema de inteligencia artificial de alto riesgo tomar 
las medidas adecuadas (t

---
## Implementación del recuperador BM25

In [6]:
STOPWORDS_ES = {
    "de", "la", "el", "en", "y", "a", "los", "las", "se", "del",
    "que", "un", "una", "con", "por", "para", "es", "al", "lo",
    "su", "sus", "o", "le", "como", "más", "pero", "sus", "no",
    "ha", "me", "si", "ya", "sea", "así", "este", "esta", "estos",
    "estas", "ese", "esa", "esos", "esas", "ser", "han", "será",
    "cuáles", "cuales", "son", "según", "segun", "qué", "que",
    "cómo", "como", "dónde", "donde", "cuándo", "cuando",
    "cuál", "cual", "quién", "quien", "quiénes", "quienes",
    "hay", "debe", "deben", "puede", "pueden", "tiene", "tienen",
    "dicho", "dichos", "dicha", "dichas", "mismo", "misma",
    "tal", "tales", "cada", "entre", "sobre", "ante", "bajo",
    "tras", "sin", "hasta", "desde", "hacia", "durante",
    "mediante", "contra", "salvo", "excepto", "sino",
    "también", "tampoco", "además", "incluso", "aunque",
    "mientras", "porque", "pues", "sino", "ni",
}

def normalizar_acentos(texto: str) -> str:
    """
    Elimina los diacríticos (acentos, tildes) de un texto mediante
    descomposición canónica Unicode (NFD) y filtrado de marcas de
    combinación (categoría 'Mn').
    Ejemplo: 'artículo' → 'articulo', 'obligación' → 'obligacion'
    """
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

def tokenizar_juridico(texto: str) -> list[str]:
    texto = normalizar_acentos(texto)
    texto = texto.lower()
    texto = re.sub(r"[^\w\s\-]", " ", texto)
    tokens = texto.split()
    tokens = [
        t for t in tokens
        if t not in STOPWORDS_ES and len(t) >= 3
    ]
    return tokens

# CONSTRUCCIÓN DEL ÍNDICE BM25
logger.info("Construyendo índice BM25 sobre el corpus...")
t0 = time.time()

corpus_tokenizado = [tokenizar_juridico(f.page_content) for f in fragmentos]
indice_bm25 = BM25Okapi(corpus_tokenizado)

t_bm25 = time.time() - t0
logger.info(f"Índice BM25 construido en {t_bm25:.2f}s sobre {len(corpus_tokenizado)} documentos")

# FUNCIÓN DE BÚSQUEDA BM25
def busqueda_bm25(
    consulta: str,
    indice: BM25Okapi,
    fragmentos: list[Document],
    k: int = 5
) -> list[dict]:
    tokens_consulta = tokenizar_juridico(consulta)
    puntuaciones = indice.get_scores(tokens_consulta)

    # Índices de los k documentos con mayor puntuación
    indices_top_k = np.argsort(puntuaciones)[::-1][:k]

    return [
        {
            "contenido" : fragmentos[idx].page_content,
            "metadatos" : fragmentos[idx].metadata,
            "score"     : float(puntuaciones[idx]),
            "indice"    : int(idx),
            "ranking"   : i + 1
        }
        for i, idx in enumerate(indices_top_k)
    ]


print(f"Índice BM25 construido.")
print(f"  Documentos indexados : {len(corpus_tokenizado)}")
print(f"  Tiempo de indexación : {t_bm25:.2f}s")
print(f"  Ejemplo de tokens    : {corpus_tokenizado[0][:15]}")

2026-06-13 15:04:33,357 [INFO] Construyendo índice BM25 sobre el corpus...
2026-06-13 15:04:34,117 [INFO] Índice BM25 construido en 0.76s sobre 5192 documentos


Índice BM25 construido.
  Documentos indexados : 5192
  Tiempo de indexación : 0.76s
  Ejemplo de tokens    : ['gestion', 'riesgo', 'evaluacion', 'impacto', 'tratamientos', 'datos', 'personales', 'junio', '2021', 'obra', 'licencia', 'creative', 'commons', 'atribucion-nocomercial-compartirigual', 'internacional']


---
## Verificación del recuperador BM25

In [7]:
RUTA_BM25 = VECTOR_DIR / "indice_bm25.pkl"
# Consulta de prueba con el mismo texto que la búsqueda semántica
resultados_bm25 = busqueda_bm25(
    CONSULTA_PRUEBA, indice_bm25, fragmentos, K_RESULTADOS
)

print(f"Consulta: '{CONSULTA_PRUEBA}'")
print(f"Tokens BM25: {tokenizar_juridico(CONSULTA_PRUEBA)}")

print(f"\n{'='*60}")
print(f"RESULTADOS BM25 (Recuperación Léxica)")
print(f"{'='*60}")

for r in resultados_bm25:
    articulos = r['metadatos'].get('articulos', 'N/A')
    fuente    = r['metadatos'].get('source', 'N/A')
    pagina    = r['metadatos'].get('page', 'N/A')
    print(f"\n  [{r['ranking']}] Score BM25: {r['score']:.4f}")
    print(f"       Fuente   : {fuente}")
    print(f"       Artículos: {articulos}")
    print(f"       Página   : {pagina}")
    print(f"       Contenido: {r['contenido'][:200]}...")


if RUTA_BM25.exists():
    logger.info("Índice BM25 encontrado en disco. Cargando...")
    with open(RUTA_BM25, "rb") as f:
        bm25_data = pickle.load(f)
    indice_bm25              = bm25_data["indice"]
    corpus_tokenizado = bm25_data["corpus_tokenizado"]
    logger.info(f"Índice BM25 cargado: {len(corpus_tokenizado)} documentos.")
else:
    logger.info("Índice BM25 no encontrado. Construyendo desde cero...")
    corpus_tokenizado = [
        tokenizar_juridico(f.page_content) for f in fragmentos
    ]
    indice_bm25 = BM25Okapi(corpus_tokenizado)
    with open(RUTA_BM25, "wb") as f:
        pickle.dump({
            "indice"            : indice_bm25,
            "corpus_tokenizado" : corpus_tokenizado,
            "n_stopwords"       : len(STOPWORDS_ES),
            "version"           : "v2"
        }, f)
    logger.info(f"Índice BM25 persistido: {RUTA_BM25.stat().st_size / 1024**2:.2f} MB")

2026-06-13 15:04:37,676 [INFO] Índice BM25 no encontrado. Construyendo desde cero...


Consulta: '¿Cuáles son las obligaciones del proveedor de un sistema de IA de alto riesgo según el AI Act?'
Tokens BM25: ['obligaciones', 'proveedor', 'sistema', 'alto', 'riesgo', 'act']

RESULTADOS BM25 (Recuperación Léxica)

  [1] Score BM25: 16.5859
       Fuente   : AESIA/14-guia-gestion-de-incidentes.pdf
       Artículos: ['26']
       Página   : 17
       Contenido: Esta es una medida global, no ejemplificable mediante un caso de uso. 
 
 
A qué apartados aplica esta medida 
• Lo que debe hacer el proveedor. 
 
4.2.2 Contacto con la Autoridad de Vigilancia del Me...

  [2] Score BM25: 16.0752
       Fuente   : AESIA/12-guia-de-registros.pdf
       Artículos: ['12']
       Página   : 12
       Contenido: 12 
 
4. Registros: actores y contenido 
4.1. Agentes responsables de los registros 
Las obligaciones asociadas a los sistemas de IA de alto riesgo en relación con la 
conservación de los registros, a...

  [3] Score BM25: 15.4815
       Fuente   : AESIA/01-guia-introductoria-al-re

2026-06-13 15:04:38,543 [INFO] Índice BM25 persistido: 7.24 MB


---
## Implementación de Reciprocal Rank Fusion (RRF)

RRF combina los rankings de múltiples recuperadores asignando una puntuación
inversamente proporcional al rango de cada fragmento en cada lista.
La fórmula es: `RRF(d) = Σ 1 / (k + rank(d))` donde k=60 es la constante estándar.

Se implementan **dos variantes** de recuperación híbrida:
- **Híbrido Baseline**: BM25 + MiniLM-L12
- **Híbrido Mejorado**: BM25 + bge-m3

In [8]:
def reciprocal_rank_fusion(
    lista_rankings: list[list[dict]],
    fragmentos: list[Document],
    k_rrf: int = 60,
    top_n: int = 5
) -> list[dict]:
    puntuaciones_rrf = {}

    for ranking in lista_rankings:
        for posicion, resultado in enumerate(ranking):
            idx = resultado["indice"]
            puntuacion = 1.0 / (k_rrf + posicion + 1)
            puntuaciones_rrf[idx] = puntuaciones_rrf.get(idx, 0.0) + puntuacion

    # Ordenación por puntuación RRF descendente
    indices_ordenados = sorted(
        puntuaciones_rrf.keys(),
        key=lambda idx: puntuaciones_rrf[idx],
        reverse=True
    )[:top_n]

    return [
        {
            "contenido"   : fragmentos[idx].page_content,
            "metadatos"   : fragmentos[idx].metadata,
            "score_rrf"   : puntuaciones_rrf[idx],
            "indice"      : idx,
            "ranking"     : i + 1
        }
        for i, idx in enumerate(indices_ordenados)
    ]

mapa_chunk_id = {f.metadata["chunk_id"]: i for i, f in enumerate(fragmentos)}

def enriquecer_con_indice(resultados_chroma: list[dict]) -> list[dict]:
    """Añade el campo 'indice' a los resultados de ChromaDB usando chunk_id."""
    for r in resultados_chroma:
        chunk_id = r["metadatos"].get("chunk_id")
        # chunk_id puede estar almacenado como string en ChromaDB
        if isinstance(chunk_id, str):
            chunk_id = int(chunk_id)
        r["indice"] = mapa_chunk_id.get(chunk_id, -1)
    return resultados_chroma


# Enriquecimiento de resultados semánticos con índice
resultados_baseline_idx = enriquecer_con_indice(resultados_baseline)
resultados_bge_m3_idx   = enriquecer_con_indice(resultados_bge_m3)
resultados_bm25_idx     = resultados_bm25

print("Función RRF")
print(f"  Constante k_rrf     : 60 (estándar de la literatura)")
print(f"  Mapa chunk_id→índice: {len(mapa_chunk_id)} entradas")
print(f"  Variantes híbridas  : Baseline (BM25+MiniLM) | Mejorado (BM25+bge-m3)")

Función RRF
  Constante k_rrf     : 60 (estándar de la literatura)
  Mapa chunk_id→índice: 5192 entradas
  Variantes híbridas  : Baseline (BM25+MiniLM) | Mejorado (BM25+bge-m3)


---
## Prueba del sistema

In [9]:
# RECUPERACIÓN HÍBRIDA — BASELINE (BM25 + MiniLM)
# Se amplía k a 20 para dar más candidatos al fusionador RRF
K_CANDIDATOS = 20

resultados_sem_baseline_amplios = busqueda_semantica(
    CONSULTA_PRUEBA, modelo_baseline, coleccion_baseline, K_CANDIDATOS
)
resultados_bm25_amplios = busqueda_bm25(
    CONSULTA_PRUEBA, indice_bm25, fragmentos, K_CANDIDATOS
)

resultados_sem_baseline_amplios = enriquecer_con_indice(resultados_sem_baseline_amplios)

hibrido_baseline = reciprocal_rank_fusion(
    lista_rankings=[resultados_sem_baseline_amplios, resultados_bm25_amplios],
    fragmentos=fragmentos,
    k_rrf=60,
    top_n=K_RESULTADOS
)

# RECUPERACIÓN HÍBRIDA — BM25 + bge-m3
resultados_sem_bge_m3_amplios = busqueda_semantica(
    CONSULTA_PRUEBA, modelo_bge_m3, coleccion_bge_m3, K_CANDIDATOS
)
resultados_sem_bge_m3_amplios = enriquecer_con_indice(resultados_sem_bge_m3_amplios)

hibrido_bge_m3 = reciprocal_rank_fusion(
    lista_rankings=[resultados_sem_bge_m3_amplios, resultados_bm25_amplios],
    fragmentos=fragmentos,
    k_rrf=60,
    top_n=K_RESULTADOS
)

def imprimir_hibrido(resultados: list[dict], nombre: str):
    print(f"\n{'='*60}")
    print(f"RECUPERACIÓN HÍBRIDA — {nombre}")
    print(f"{'='*60}")
    for r in resultados:
        articulos = r['metadatos'].get('articulos', 'N/A')
        fuente    = r['metadatos'].get('source', 'N/A')
        print(f"\n  [{r['ranking']}] Score RRF: {r['score_rrf']:.6f}")
        print(f"       Fuente   : {fuente}")
        print(f"       Artículos: {articulos}")
        print(f"       Contenido: {r['contenido'][:200]}...")

imprimir_hibrido(hibrido_baseline, "BM25 + MiniLM-L12")
imprimir_hibrido(hibrido_bge_m3,   "BM25 + bge-m3")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


RECUPERACIÓN HÍBRIDA — BM25 + MiniLM-L12

  [1] Score RRF: 0.028986
       Fuente   : AESIA/08-guia-transparencia.pdf
       Artículos: []
       Contenido: la identidad y los datos de contacto del proveedor y, en su caso, de su 
representante autorizado; 
 
Qué entendemos 
Dado que son sistemas de IA de alto riesgo, pretende a segurar el servicio de sopo...

  [2] Score RRF: 0.027047
       Fuente   : AESIA/01-guia-introductoria-al-reglamento-de-ia-1770802981.pdf
       Artículos: []
       Contenido: Reglamento si se producen alguna de las siguientes circunstancias: 
• Ponen su nombre o marca en un sistema de IA de alto riesgo que ya se ha 
introducido en el mercado o puesto en servicio, sin perju...

  [3] Score RRF: 0.016393
       Fuente   : AESIA/11-guia-ciberseguridad.pdf
       Artículos: []
       Contenido: 49 
 
Si el formato de uso y comercialización del sistema de IA contemplase que el responsable 
del despliegue dispusiese del sistema de IA como un activo de sus propias 

---
## Comparacion: semántico vs léxico vs híbrido

In [10]:
CONSULTAS_AUDITORIA = [
    "¿Qué sistemas de IA se consideran de alto riesgo?",
    "Obligaciones del proveedor en materia de transparencia",
    "Artículo 10 requisitos de datos de entrenamiento",
    "¿Qué es un sistema de IA de propósito general?",
    "Sanciones y multas por incumplimiento del reglamento",
]

K_AUDITORIA  = 10
TOP_AUDITORIA = 5

print("=" * 65)
print("Comparación")
print("=" * 65)
print(f"{'Consulta':<45} {'SemB':>6} {'SemM':>6} {'BM25':>6} {'HybB':>6} {'HybM':>6}")
print(f"{'':->45} {'----':>6} {'----':>6} {'----':>6} {'----':>6} {'----':>6}")
print("(SemB=Semántico Baseline, SemM=Semántico bge-m3,")
print(" BM25=Léxico, HybB=Híbrido Baseline, HybM=Híbrido bge-m3)")
print("-" * 65)

tabla_resultados = []

for consulta in CONSULTAS_AUDITORIA:
    # Búsqueda con cada recuperador
    res_sem_base = busqueda_semantica(
        consulta, modelo_baseline, coleccion_baseline, K_AUDITORIA
    )
    res_sem_bge  = busqueda_semantica(
        consulta, modelo_bge_m3, coleccion_bge_m3, K_AUDITORIA
    )
    res_bm25     = busqueda_bm25(
        consulta, indice_bm25, fragmentos, K_AUDITORIA
    )

    # Enriquecimiento con índice
    res_sem_base = enriquecer_con_indice(res_sem_base)
    res_sem_bge  = enriquecer_con_indice(res_sem_bge)

    # Fusión RRF
    res_hyb_base = reciprocal_rank_fusion(
        [res_sem_base, res_bm25], fragmentos, top_n=TOP_AUDITORIA
    )
    res_hyb_bge  = reciprocal_rank_fusion(
        [res_sem_bge, res_bm25], fragmentos, top_n=TOP_AUDITORIA
    )

    # Extracción de artículos del AI Act en top-5 de cada recuperador
    def extraer_articulos(resultados):
        arts = set()
        for r in resultados[:TOP_AUDITORIA]:
            meta = r.get("metadatos", {})
            arts_raw = meta.get("articulos", "")
            if isinstance(arts_raw, str) and arts_raw:
                arts.update(arts_raw.split(", "))
            elif isinstance(arts_raw, list):
                arts.update(str(a) for a in arts_raw)
        return sorted(arts)

    arts_semb = extraer_articulos(res_sem_base)
    arts_semm = extraer_articulos(res_sem_bge)
    arts_bm25 = extraer_articulos(res_bm25)
    arts_hybb = extraer_articulos(res_hyb_base)
    arts_hybm = extraer_articulos(res_hyb_bge)

    consulta_corta = consulta[:43] + ".." if len(consulta) > 43 else consulta
    print(f"{consulta_corta:<45} "
          f"{str(arts_semb):>6} "
          f"{str(arts_semm):>6} "
          f"{str(arts_bm25):>6} "
          f"{str(arts_hybb):>6} "
          f"{str(arts_hybm):>6}")

    tabla_resultados.append({
        "consulta"    : consulta,
        "sem_baseline": arts_semb,
        "sem_bge_m3"  : arts_semm,
        "bm25"        : arts_bm25,
        "hyb_baseline": arts_hybb,
        "hyb_bge_m3"  : arts_hybm,
    })

print("=" * 65)
print("\nInterpretación:")
print("  - Artículos únicos en híbrido no presentes en semántico puro")
print("    indican que BM25 aporta cobertura terminológica adicional.")
print("  - La convergencia entre HybB y HybM valida la robustez del RRF")
print("    independientemente del modelo de embeddings subyacente.")

Comparación
Consulta                                        SemB   SemM   BM25   HybB   HybM
---------------------------------------------   ----   ----   ----   ----   ----
(SemB=Semántico Baseline, SemM=Semántico bge-m3,
 BM25=Léxico, HybB=Híbrido Baseline, HybM=Híbrido bge-m3)
-----------------------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

¿Qué sistemas de IA se consideran de alto r..     []  ['9']     []     []  ['9']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

¿Qué es un sistema de IA de propósito gener..     []  ['3']     []     []  ['3']


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Sanciones y multas por incumplimiento del r..     [] ['58', '83'] ['58', '83'] ['58', '83'] ['58', '83']

Interpretación:
  - Artículos únicos en híbrido no presentes en semántico puro
    indican que BM25 aporta cobertura terminológica adicional.
  - La convergencia entre HybB y HybM valida la robustez del RRF
    independientemente del modelo de embeddings subyacente.


---
## Resumen

In [11]:
estado_fase2 = {
    "modelos": {
        "baseline": {
            "id"             : MODELO_BASELINE_ID,
            "dimension"      : int(embeddings_baseline.shape[1]),
            "chroma_dir"     : str(CHROMA_BASELINE_DIR),
            "coleccion"      : "ai_act_baseline",
            "batch_size"     : BATCH_SIZE_BASELINE,
        },
        "bge_m3": {
            "id"             : MODELO_BGE_M3_ID,
            "dimension"      : int(embeddings_bge_m3.shape[1]),
            "chroma_dir"     : str(CHROMA_BGE_M3_DIR),
            "coleccion"      : "ai_act_bge_m3",
            "batch_size"     : BATCH_SIZE_BGE_M3,
        }
    },
    "bm25": {
        "stopwords_count" : len(STOPWORDS_ES),
        "documentos"      : len(corpus_tokenizado),
    },
    "rrf": {
        "k_rrf"           : 60,
        "k_candidatos"    : K_CANDIDATOS,
    },
    "corpus": {
        "fichero"         : str(RUTA_CORPUS),
        "total_fragmentos": len(fragmentos),
    }
}

RUTA_ESTADO_FASE2 = BASE_DIR / "fase2_estado.json"
with open(RUTA_ESTADO_FASE2, "w", encoding="utf-8") as f:
    json.dump(estado_fase2, f, ensure_ascii=False, indent=2)

# -------------------------------------------------------------------
# RESUMEN FINAL
# -------------------------------------------------------------------
print("=" * 65)
print("RESUMEN")
print("=" * 65)
print(f"  Fragmentos indexados       : {len(fragmentos)}")
print(f"  Modelo baseline            : {MODELO_BASELINE_ID}")
print(f"    Dimensión del vector     : {embeddings_baseline.shape[1]}")
print(f"    Persistencia             : {CHROMA_BASELINE_DIR.name}")
print(f"  Modelo mejorado            : {MODELO_BGE_M3_ID}")
print(f"    Dimensión del vector     : {embeddings_bge_m3.shape[1]}")
print(f"    Persistencia             : {CHROMA_BGE_M3_DIR.name}")
print(f"  Recuperador léxico         : BM25 (rank-bm25)")
print(f"  Fusión de rankings         : Reciprocal Rank Fusion (k=60)")
print(f"  Estado Fase 2 guardado en  : {RUTA_ESTADO_FASE2.name}")
print("")
print("  Artefactos listos para la Fase 3 (Orquestación Generativa):")
print(f"    - {CHROMA_BASELINE_DIR.name}/  (ChromaDB baseline)")
print(f"    - {CHROMA_BGE_M3_DIR.name}/    (ChromaDB bge-m3)")
print(f"    - fase2_estado.json             (configuración de índices)")
print("")
print("  SIGUIENTE PASO — Fase 3: Orquestación Generativa (LLM + RAG)")
print("    - Carga de LLM local (Mistral 7B / LLaMA 3.1 8B) via Ollama")
print("    - Construcción del prompt con contexto normativo recuperado")
print("    - Sistema de citación de artículos en las respuestas")
print("    - Evaluación RAGAS sobre conjunto de prueba")
print("=" * 65)

RESUMEN
  Fragmentos indexados       : 5192
  Modelo baseline            : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
    Dimensión del vector     : 384
    Persistencia             : chroma_baseline
  Modelo mejorado            : BAAI/bge-m3
    Dimensión del vector     : 1024
    Persistencia             : chroma_bge_m3
  Recuperador léxico         : BM25 (rank-bm25)
  Fusión de rankings         : Reciprocal Rank Fusion (k=60)
  Estado Fase 2 guardado en  : fase2_estado.json

  Artefactos listos para la Fase 3 (Orquestación Generativa):
    - chroma_baseline/  (ChromaDB baseline)
    - chroma_bge_m3/    (ChromaDB bge-m3)
    - fase2_estado.json             (configuración de índices)

  SIGUIENTE PASO — Fase 3: Orquestación Generativa (LLM + RAG)
    - Carga de LLM local (Mistral 7B / LLaMA 3.1 8B) via Ollama
    - Construcción del prompt con contexto normativo recuperado
    - Sistema de citación de artículos en las respuestas
    - Evaluación RAGAS sobre conjunto de